<a href="https://colab.research.google.com/github/cylin577/Image2Audio/blob/main/i2agpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Install dependency
!pip install gradio
!pip install soundfile
!pip install pillow
!pip install cupy-cuda12x

In [ ]:
# @title Main
import cupy as cp  # Using CuPy instead of NumPy for GPU acceleration
from PIL import Image
import soundfile as sf
import gradio as gr
from scipy.io.wavfile import write

# English language support
LANGUAGE = {
    "title": "Image-Audio Encoder/Decoder",
    "encode_button": "Select Image to Encode",
    "decode_button": "Select Audio to Decode",
    "success_encode": "Image encoded to audio.",
    "success_decode": "Audio decoded back to image.",
    "error_length": "Decoded data length does not match expected length.",
}

def encode_image_to_audio(image):
    # Process the image using CuPy (GPU acceleration)
    data = cp.array(image)  # Convert image to CuPy array
    height, width, _ = data.shape
    audio_data = data / 255.0 * 2 - 1  # Normalize data to [-1, 1]
    audio_data_flattened = audio_data.flatten()  # Flatten the data for audio encoding
    size_info = cp.array([height, width], dtype=cp.int32)  # Store image size
    audio_data_with_size = cp.concatenate([size_info, audio_data_flattened])

    audio_path = "encoded_audio.wav"
    # Save the audio using NumPy array (ensure compatibility)
    write(audio_path, 44100, cp.asnumpy(audio_data_with_size).astype(cp.float32))
    
    return LANGUAGE["success_encode"], audio_path

def decode_audio_to_image(audio):
    # Read the audio data using soundfile
    audio_data, sample_rate = sf.read(audio)
    height = int(audio_data[0])  # Retrieve height
    width = int(audio_data[1])   # Retrieve width
    data = cp.array(audio_data[2:])  # Use CuPy for data processing
    expected_length = height * width * 3  # Expected data length

    # Check if the length of the data matches the expected length
    if len(data) != expected_length:
        return LANGUAGE["error_length"], None

    # Reconstruct the image data
    img_data = ((data + 1) / 2 * 255).astype(cp.uint8)  # Convert back to image range
    img_data = img_data.reshape((height, width, 3))  # Reshape into an image
    
    img = Image.fromarray(cp.asnumpy(img_data))  # Convert to NumPy for PIL processing
    image_path = "decoded_image.png"
    img.save(image_path)

    return LANGUAGE["success_decode"], image_path

# Gradio interface setup
def interface():
    with gr.Blocks() as demo:
        gr.Markdown(f"## {LANGUAGE['title']}")
        with gr.Row():
            with gr.Column():
                # Encoding section
                img_input = gr.Image(label=LANGUAGE["encode_button"])
                audio_output = gr.File(label="Encoded Audio File")
                encode_button = gr.Button(LANGUAGE["encode_button"])
                
                # Bind encode button to the encoding function
                encode_button.click(encode_image_to_audio, inputs=[img_input], outputs=[gr.Text(label="Outputs"), audio_output])

            with gr.Column():
                # Decoding section
                audio_input = gr.Audio(label=LANGUAGE["decode_button"], type="filepath")
                img_output = gr.Image(label="Decoded Image")
                decode_button = gr.Button(LANGUAGE["decode_button"])
                
                # Bind decode button to the decoding function
                decode_button.click(decode_audio_to_image, inputs=[audio_input], outputs=[gr.Text(label="Outputs"), img_output])
    
    return demo

# Run the app with `share=True` and `debug=True` enabled
interface().launch(share=True, debug=True)
